In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from collections import defaultdict, Counter
import time


qc = QuantumCircuit.from_qasm_file("P7_rolling_ridge.qasm")
print(f"Qubits: {qc.num_qubits}, Depth: {qc.depth()}, Gates: {qc.count_ops()}")

# find connected components (independent qubit groups)
interactions = set()
for instr in qc.data:
    if len(instr.qubits) == 2:
        q1 = instr.qubits[0]._index
        q2 = instr.qubits[1]._index
        interactions.add((min(q1,q2), max(q1,q2)))

graph = defaultdict(set)
for q1, q2 in interactions:
    graph[q1].add(q2)
    graph[q2].add(q1)

visited = set()
components = []
for q in range(qc.num_qubits):
    if q not in visited:
        component = []
        stack = [q]
        while stack:
            node = stack.pop()
            if node not in visited:
                visited.add(node)
                component.append(node)
                stack.extend(graph[node] - visited)
        components.append(sorted(component))

print(f"\nFound {len(components)} independent components:")
for i, comp in enumerate(components):
    print(f"  Part {i+1}: {len(comp)} qubits → {comp}")

# simulate each component separately
simulator = AerSimulator(
    method="matrix_product_state",
    matrix_product_state_max_bond_dimension=512,
)

full_peak = ['0'] * qc.num_qubits

for part_idx, component in enumerate(components):
    print(f"\nSimulating part {part_idx+1} ({len(component)} qubits)...")
    
    # extract subcircuit
    qubit_map = {q: i for i, q in enumerate(component)}
    sub_qc = QuantumCircuit(len(component))
    
    for instr in qc.data:
        qubits = [b._index for b in instr.qubits]
        if all(q in qubit_map for q in qubits):
            mapped = [qubit_map[q] for q in qubits]
            sub_qc.append(instr.operation, mapped)
    
    print(f"  Subcircuit: {sub_qc.count_ops()}")
    
    # try statevector for small parts
    if len(component) <= 28:
        print(f"  Using statevector (small enough)...")
        sim = AerSimulator(method="statevector")
    else:
        sim = simulator
    
    sub_qc.measure_all()
    start = time.time()
    
    all_counts = Counter()
    runs = 1 if len(component) <= 28 else 3
    for i in range(runs):
        job = sim.run(sub_qc, shots=4096)
        counts = job.result().get_counts()
        all_counts.update(counts)
    
    total = sum(all_counts.values())
    peak = max(all_counts, key=all_counts.get)
    print(f"  Peak: {peak} ({all_counts[peak]/total:.4%}) | Time: {time.time()-start:.0f}s")
    
    # place bits back in correct positions
    for bit_idx, qubit in enumerate(component):
        full_peak[qubit] = peak[-(bit_idx+1)] 

# combine results
final_peak = ''.join(full_peak[::-1])
print(f"\n{'='*60}")
print(f"Final combined peak:\n{final_peak}")
print(f"{'='*60}")

# reconstruct 
full_peak = ['0'] * qc.num_qubits

part1_qubits = [0, 1, 2, 4, 6, 7, 8, 11, 13, 14, 15, 18, 20, 24, 26, 30, 31, 34, 37, 39]
part1_peak = "10100000110110110100"

part2_qubits = [3, 5, 9, 10, 12, 16, 17, 19, 21, 22, 23, 25, 27, 28, 29, 32, 33, 35, 36, 38, 40, 41]
part2_peak = "1011001001010010010100"

# qiskit measures in reverse order
for bit_idx, qubit in enumerate(part1_qubits):
    full_peak[qubit] = part1_peak[-(bit_idx+1)]

for bit_idx, qubit in enumerate(part2_qubits):
    full_peak[qubit] = part2_peak[-(bit_idx+1)]

final = ''.join(full_peak[::-1])
print(f"Final peak:\n{final}")
print(f"Length: {len(final)}")

Qubits: 42, Depth: 93, Gates: OrderedDict({'u3': 1939, 'cz': 949})

Found 2 independent components:
  Part 1: 20 qubits → [0, 1, 2, 4, 6, 7, 8, 11, 13, 14, 15, 18, 20, 24, 26, 30, 31, 34, 37, 39]
  Part 2: 22 qubits → [3, 5, 9, 10, 12, 16, 17, 19, 21, 22, 23, 25, 27, 28, 29, 32, 33, 35, 36, 38, 40, 41]

Simulating part 1 (20 qubits)...
  Subcircuit: OrderedDict({'u3': 927, 'cz': 454})
  Using statevector (small enough)...
  Peak: 10100000110110110100 (48.9014%) | Time: 2s

Simulating part 2 (22 qubits)...
  Subcircuit: OrderedDict({'u3': 1012, 'cz': 495})
  Using statevector (small enough)...
  Peak: 1011001001010010010100 (49.3164%) | Time: 6s

Final combined peak:
101101010100001000100011001011101011000100
Final peak:
101101010100001000100011001011101011000100
Length: 42
